# The file will convert the pre-processed data to modelling data

### 1. Load the data and delete features can not be used for prediction: like name, coordinates...

In [1]:
# import needed libraries
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score
import statsmodels.api as sm # pip install statsmodels
from sklearn.model_selection import train_test_split

In [2]:
# select needed features for prediction rental price
data_2023 = pd.read_csv("../data/curated/final_data/merged_data_2023.csv")
property_features = ['num_bedroom', 'num_bathroom', 'num_parking', 'rental_price'] # sa2 is deleted, if needed, plz add back!
distance_features = ['log_school_distance', 'log_station_distance', 'log_hospital_distance', 'log_mall_distance', 'log_park_distance', 'log_CBD_distance']
suburb_features = ['personal_income', 'pop_density', 'offence_count']
features_list = suburb_features + distance_features + property_features
origin_features_list = features_list
modelling_data = data_2023[features_list]
modelling_data

,personal_income,pop_density,offence_count,log_school_distance,log_station_distance,log_hospital_distance,log_mall_distance,log_park_distance,log_CBD_distance,num_bedroom,num_bathroom,num_parking,rental_price
0,72706.625309,965.216952,17.333333,0.640804,2.475412,1.906040,1.239505,1.345502,4.578730,4,2,2.0,575.0
1,72706.625309,1746.760558,17.333333,-0.348952,1.681479,1.438925,1.052926,1.716843,4.559835,4,2,2.0,560.0
2,72706.625309,965.216952,17.333333,0.095820,2.187960,1.847653,1.673884,1.622239,4.586220,2,2,1.0,490.0
3,72706.625309,965.216952,17.333333,-0.393160,2.092679,1.736704,0.793410,1.183953,4.578058,4,2,1.0,540.0
4,70490.898811,151.921192,31.000000,0.799813,2.315778,2.017863,1.375422,0.115940,4.602687,4,2,2.0,520.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8214,100462.815420,1773.237842,25.000000,-2.228826,0.138021,0.374363,0.344505,1.026702,3.826546,2,1,0.0,630.0
8215,100462.815420,1773.237842,25.000000,-1.427862,0.754430,1.122437,0.873918,1.320736,3.823240,4,3,2.0,730.0
8216,100462.815420,1773.237842,25.000000,-0.199202,1.034856,1.025788,0.821525,1.484705,3.805227,3,1,2.0,450.0
8217,100462.815420,1773.237842,25.000000,-1.237029,0.614537,0.623725,0.746158,1.316399,3.810149,1,1,0.0,300.0


In [3]:
# Add a constant term (intercept term)
modelling_data['intercept'] = 1
modelling_data = modelling_data[['intercept'] + features_list]
modelling_data

/tmp/ipykernel_8969/3930595328.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  modelling_data['intercept'] = 1


,intercept,personal_income,pop_density,offence_count,log_school_distance,log_station_distance,log_hospital_distance,log_mall_distance,log_park_distance,log_CBD_distance,num_bedroom,num_bathroom,num_parking,rental_price
0,1,72706.625309,965.216952,17.333333,0.640804,2.475412,1.906040,1.239505,1.345502,4.578730,4,2,2.0,575.0
1,1,72706.625309,1746.760558,17.333333,-0.348952,1.681479,1.438925,1.052926,1.716843,4.559835,4,2,2.0,560.0
2,1,72706.625309,965.216952,17.333333,0.095820,2.187960,1.847653,1.673884,1.622239,4.586220,2,2,1.0,490.0
3,1,72706.625309,965.216952,17.333333,-0.393160,2.092679,1.736704,0.793410,1.183953,4.578058,4,2,1.0,540.0
4,1,70490.898811,151.921192,31.000000,0.799813,2.315778,2.017863,1.375422,0.115940,4.602687,4,2,2.0,520.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8214,1,100462.815420,1773.237842,25.000000,-2.228826,0.138021,0.374363,0.344505,1.026702,3.826546,2,1,0.0,630.0
8215,1,100462.815420,1773.237842,25.000000,-1.427862,0.754430,1.122437,0.873918,1.320736,3.823240,4,3,2.0,730.0
8216,1,100462.815420,1773.237842,25.000000,-0.199202,1.034856,1.025788,0.821525,1.484705,3.805227,3,1,2.0,450.0
8217,1,100462.815420,1773.237842,25.000000,-1.237029,0.614537,0.623725,0.746158,1.316399,3.810149,1,1,0.0,300.0


In [4]:
selected_features = modelling_data.drop(['rental_price'], axis=1)
selected_features

,intercept,personal_income,pop_density,offence_count,log_school_distance,log_station_distance,log_hospital_distance,log_mall_distance,log_park_distance,log_CBD_distance,num_bedroom,num_bathroom,num_parking
0,1,72706.625309,965.216952,17.333333,0.640804,2.475412,1.906040,1.239505,1.345502,4.578730,4,2,2.0
1,1,72706.625309,1746.760558,17.333333,-0.348952,1.681479,1.438925,1.052926,1.716843,4.559835,4,2,2.0
2,1,72706.625309,965.216952,17.333333,0.095820,2.187960,1.847653,1.673884,1.622239,4.586220,2,2,1.0
3,1,72706.625309,965.216952,17.333333,-0.393160,2.092679,1.736704,0.793410,1.183953,4.578058,4,2,1.0
4,1,70490.898811,151.921192,31.000000,0.799813,2.315778,2.017863,1.375422,0.115940,4.602687,4,2,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8214,1,100462.815420,1773.237842,25.000000,-2.228826,0.138021,0.374363,0.344505,1.026702,3.826546,2,1,0.0
8215,1,100462.815420,1773.237842,25.000000,-1.427862,0.754430,1.122437,0.873918,1.320736,3.823240,4,3,2.0
8216,1,100462.815420,1773.237842,25.000000,-0.199202,1.034856,1.025788,0.821525,1.484705,3.805227,3,1,2.0
8217,1,100462.815420,1773.237842,25.000000,-1.237029,0.614537,0.623725,0.746158,1.316399,3.810149,1,1,0.0


### 2. Using BIC to delete features with less significant

In [5]:
import statsmodels.api as sm

# Set the BIC threshold
BIC_threshold = float('inf')  # Initialize with positive infinity

# Start backward elimination
while True:
    # Fit model and calculate BIC
    model = sm.OLS(modelling_data['rental_price'], selected_features).fit()
    BIC_value = model.bic
    
    # Find the feature with the maximum BIC value
    max_BIC_feature = model.pvalues.idxmax()
    
    # If the maximum BIC value is less than the current threshold, update the threshold
    if BIC_value < BIC_threshold:
        BIC_threshold = BIC_value
    # If the maximum BIC value is greater than or equal to the threshold, terminate the loop
    else:
        break
    
    # Eliminate the least significant feature
    selected_features = selected_features.drop(max_BIC_feature, axis=1)

# Print the final selected features
final_features = selected_features.columns
print("Final selected features:")
print(final_features)

Final selected features:
Index(['personal_income', 'pop_density', 'offence_count',
       'log_school_distance', 'log_station_distance', 'log_hospital_distance',
       'log_mall_distance', 'log_park_distance', 'log_CBD_distance',
       'num_bedroom', 'num_bathroom'],
      dtype='object')


In [6]:
# apply selected features to the moddelling data
cloumns_list = final_features.tolist()
features_list = cloumns_list.copy()
cloumns_list.append('rental_price')
modelling_data = data_2023[cloumns_list]
modelling_data

,personal_income,pop_density,offence_count,log_school_distance,log_station_distance,log_hospital_distance,log_mall_distance,log_park_distance,log_CBD_distance,num_bedroom,num_bathroom,rental_price
0,72706.625309,965.216952,17.333333,0.640804,2.475412,1.906040,1.239505,1.345502,4.578730,4,2,575.0
1,72706.625309,1746.760558,17.333333,-0.348952,1.681479,1.438925,1.052926,1.716843,4.559835,4,2,560.0
2,72706.625309,965.216952,17.333333,0.095820,2.187960,1.847653,1.673884,1.622239,4.586220,2,2,490.0
3,72706.625309,965.216952,17.333333,-0.393160,2.092679,1.736704,0.793410,1.183953,4.578058,4,2,540.0
4,70490.898811,151.921192,31.000000,0.799813,2.315778,2.017863,1.375422,0.115940,4.602687,4,2,520.0
...,...,...,...,...,...,...,...,...,...,...,...,...
8214,100462.815420,1773.237842,25.000000,-2.228826,0.138021,0.374363,0.344505,1.026702,3.826546,2,1,630.0
8215,100462.815420,1773.237842,25.000000,-1.427862,0.754430,1.122437,0.873918,1.320736,3.823240,4,3,730.0
8216,100462.815420,1773.237842,25.000000,-0.199202,1.034856,1.025788,0.821525,1.484705,3.805227,3,1,450.0
8217,100462.815420,1773.237842,25.000000,-1.237029,0.614537,0.623725,0.746158,1.316399,3.810149,1,1,300.0


In [7]:
# saving modelling data into CSV
modelling_data.to_csv('../data/curated/final_data/modelling_data_2023.csv', index=False) 
modelling_data = pd.read_csv('../data/curated/final_data/modelling_data_2023.csv')
modelling_data

,personal_income,pop_density,offence_count,log_school_distance,log_station_distance,log_hospital_distance,log_mall_distance,log_park_distance,log_CBD_distance,num_bedroom,num_bathroom,rental_price
0,72706.625309,965.216952,17.333333,0.640804,2.475412,1.906040,1.239505,1.345502,4.578730,4,2,575.0
1,72706.625309,1746.760558,17.333333,-0.348952,1.681479,1.438925,1.052926,1.716843,4.559835,4,2,560.0
2,72706.625309,965.216952,17.333333,0.095820,2.187960,1.847653,1.673884,1.622239,4.586220,2,2,490.0
3,72706.625309,965.216952,17.333333,-0.393160,2.092679,1.736704,0.793410,1.183953,4.578058,4,2,540.0
4,70490.898811,151.921192,31.000000,0.799813,2.315778,2.017863,1.375422,0.115940,4.602687,4,2,520.0
...,...,...,...,...,...,...,...,...,...,...,...,...
8214,100462.815420,1773.237842,25.000000,-2.228826,0.138021,0.374363,0.344505,1.026702,3.826546,2,1,630.0
8215,100462.815420,1773.237842,25.000000,-1.427862,0.754430,1.122437,0.873918,1.320736,3.823240,4,3,730.0
8216,100462.815420,1773.237842,25.000000,-0.199202,1.034856,1.025788,0.821525,1.484705,3.805227,3,1,450.0
8217,100462.815420,1773.237842,25.000000,-1.237029,0.614537,0.623725,0.746158,1.316399,3.810149,1,1,300.0
